In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from typing import TypedDict
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
 
load_dotenv()

llm = ChatOpenRouter(
    model="z-ai/glm-5.3-flash",
    temperature=0.7
)


In [5]:
from langchain_community.document_loaders import PyPDFLoader
 
loader = PyPDFLoader("attention_is_all_you_need.pdf")
docs = loader.load()


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
 
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,
                                          chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [9]:
import os
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="baai/bge-m3",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local("vector_store")

ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)


In [ ]:
retriever.invoke("What is the main idea of the paper?")

In [ ]:
from langchain_core.tools import Tool, tool

@tool
def rag_tool(query: str) -> str:
    """retrieves relevant information from the PDF DOCUMENT.
    Use this tool when the user asks factual or conceptual questions that may be answere using the stored PDF documents"""

    documents = retriever.invoke(query)

    if not documents:
        return 'no relevant information was found in the PDF'

    formatted_documents = []

    for index, document in enumerate(documents, start=1):
        source = document.metadata.get("source", "Unknown source") # nếu thiếu dùng giá trị mặc định
        page = document.metadata.get("page", "Unknown page")

        formatted_documents.append(
            f"Document {index}\n"
            f"Source: {source}\n"
            f"Page: {page}\n"
            f"Content: {document.page_content}"
    )

    return "\n\n".join(formatted_documents)


    

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

class chatsate(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: chatsate):
    """LLM NODE that may answer or request a tool call"""
    message = state["message"]
    response =llm_with_tools.invoke(message)
    return{'message':[response]}

tool_node = ToolNode(tools)

In [ ]:
# graph structure
graph = StateGraph(chatsate)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")
chatbot = graph.compile()

chatbot